# LangSmith – Observability for LangGraph + Anthropic Claude

**LangSmith** is LangChain's observability and evaluation platform. It automatically captures every LLM call, tool execution, and LangGraph node run as a structured **trace** — giving you full visibility into what your AI application is doing.

## Table of Contents

| # | Topic |
|---|-------|
| **Theory** | Why Observability? The AI Black Box Problem |
| **Setup** | Sign Up & Create LangSmith API Key (step-by-step) |
| 1 | Install & Setup |
| 2 | What is LangSmith Observability? |
| 3 | Automatic Tracing — Claude + LangGraph (zero-code) |
| 4 | `@traceable` — Custom Spans for non-LangChain code |
| 5 | Tracing the Cyclic Agent Graph (Loops) |
| 6 | Tracing Memory / MemorySaver Graph |
| 7 | Tracing HITL (Human-in-the-Loop) |
| 8 | Tracing the Loan Approval System |
| 9 | Adding Feedback & Evaluation Scores |
| 10 | LangSmith Dashboard — What to look for |
| 11 | Creating & Managing Datasets |
| 12 | Automated Evaluation with Custom Evaluators |
| 13 | Latency & Cost Monitoring |
| 14 | Error Tracking & Debugging a Failing Agent |
| 15 | Summary, Checklist & Best Practices |

> **Pre-requisite:** Sign up at [smith.langchain.com](https://smith.langchain.com) and create a project. Copy your API key to `.env`.

---

## Theory – Why Observability? The AI Black Box Problem

### The Problem Without Observability

When you build an LLM application **without** a tracing/observability tool, you face a classic "black box" problem:

```
User Input  ──►  [ ??? LLM App ??? ]  ──►  Output (wrong/slow/expensive)
                         ↑
              You have NO idea what happened in here
```

**Real problems you will hit in production:**

| Problem | Without LangSmith | With LangSmith |
|---------|------------------|----------------|
| Wrong answer | "Something is broken" | See exact prompt sent + response received |
| Agent loop runs 10x | "Why is it slow?" | Trace shows every node iteration and latency |
| Costs explode | "Where are tokens going?" | Token usage tracked per run, per project |
| A node crashes | Stack trace in your terminal | Error captured in trace tree with full context |
| User complains | Can't reproduce it | Replay the exact run from LangSmith |
| Model upgrade breaks output | Hard to compare | Side-by-side experiment comparison |

---

### The Three Pillars of Observability

| Pillar | What it means | LangSmith feature |
|--------|--------------|-------------------|
| **Logs** | Record of every event | Run inputs/outputs captured per node |
| **Metrics** | Quantified measurements | Token count, latency, cost per run |
| **Traces** | End-to-end execution flow | Tree of parent → child spans across graph nodes |

---

### What is LangSmith?

LangSmith is LangChain's **official observability, evaluation, and testing platform** built specifically for LLM applications. Unlike generic APM tools (Datadog, New Relic), LangSmith understands AI-native concepts:

- LangChain chains and LangGraph nodes
- LLM prompts, completions, and token costs
- Tool call arguments and results
- Memory checkpoints and HITL pauses

### Core LangSmith Capabilities

| Capability | What you do | Benefit |
|------------|------------|---------|
| **Tracing** | Set `LANGCHAIN_TRACING_V2=true` | See every run as a structured tree |
| **Evaluation** | Write scoring functions | Automatic quality checks at scale |
| **Datasets** | Save example inputs/outputs | Regression testing with real data |
| **Experiments** | Run batch eval across dataset | Compare models or prompts objectively |
| **Monitoring** | View dashboards | Latency, cost, error-rate trends |
| **Feedback** | Attach human scores to runs | Build gold-standard eval sets |
| **Playground** | Edit prompt, re-run from trace | Fast prompt debugging |
| **Hub** | Version-control prompts | Share prompts across team |

---

### LangSmith Architecture

```
Your Python Application
    │
    ├── LangGraph / LangChain calls  ──► auto-captured (zero code change)
    ├── @traceable functions         ──► captured as child spans
    └── ls_client SDK calls          ──► manual fine-grained control
                │
                ▼  (async, non-blocking)
    ┌──────────────────────────────────────────┐
    │     LangSmith Platform (cloud)           │
    │  ┌──────────┐  ┌───────────┐             │
    │  │  Traces  │  │ Datasets  │             │
    │  └──────────┘  └───────────┘             │
    │  ┌──────────┐  ┌───────────┐             │
    │  │Evaluations│ │ Feedback  │             │
    │  └──────────┘  └───────────┘             │
    └──────────────────────────────────────────┘
```

> **Key point:** Tracing is **async and non-blocking**. It adds minimal overhead to your application — traces are buffered and sent in the background.

---

### LangSmith vs Alternatives

| Tool | LLM-native? | LangGraph support | Evaluation | Free tier |
|------|-------------|------------------|------------|-----------|
| **LangSmith** | ✅ Yes | ✅ Native | ✅ Built-in | ✅ Yes |
| Weights & Biases | Partial | ❌ No | ✅ Yes | ✅ Yes |
| MLflow | Partial | ❌ No | ✅ Yes | ✅ Self-hosted |
| Datadog APM | ❌ No | ❌ No | ❌ No | ❌ No |
| Helicone | ✅ Yes | ❌ No | Limited | ✅ Yes |
| Arize Phoenix | Partial | ❌ No | ✅ Yes | ✅ Yes |

**Choose LangSmith when:** your stack uses LangChain, LangGraph, or Anthropic/OpenAI via LangChain wrappers.

---

## Getting Started – Create Your LangSmith Account & API Key

### Step 1: Sign Up (free)

1. Open your browser → go to **https://smith.langchain.com**
2. Click **"Sign Up"**
3. Choose sign-in method: **GitHub**, **Google**, or Email + password
4. Confirm your email if required
5. You land in your **default workspace**

---

### Step 2: Create an API Key

1. Click your **profile picture** (top-right corner)
2. Go to **"Settings"**
3. Click **"API Keys"** in the left menu
4. Click **"Create API Key"**
5. Give it a name: e.g. `dev-key` or `training-key`
6. Click **"Create"**
7. ⚠️ **Copy the key immediately** — it starts with `lsv2_pt_...`
   - You **cannot view it again** after closing the dialog

---

### Step 3: Do You Need a Project?

**No — a project is not mandatory.** LangSmith has a built-in `default` project that captures all traces automatically.

However, creating a **named project** is recommended because:
- Keeps traces from different applications separate
- Easier to filter, search, and share
- Cleaner dashboards per application

#### How to create a project (optional but recommended)

1. In the left sidebar → click **"Projects"**
2. Click **"+ New Project"** (top right)
3. Enter a name: e.g. `claude-langgraph`
4. Click **"Create"**

```
smith.langchain.com
  └── Workspace
        ├── default          ← used if LANGCHAIN_PROJECT is not set
        └── claude-langgraph ← your custom project
```

If you skip this step, simply set `LANGCHAIN_PROJECT=default` in your `.env`.

---

### Step 4: Add Keys to Your `.env` File

```
# LangSmith Observability
LANGSMITH_API_KEY=lsv2_pt_your_actual_key_here
LANGCHAIN_TRACING_V2=true
LANGCHAIN_PROJECT=claude-langgraph
```

| Variable | Value | Notes |
|----------|-------|-------|
| `LANGSMITH_API_KEY` | `lsv2_pt_...` from Settings → API Keys | Required |
| `LANGCHAIN_TRACING_V2` | `true` | Required — the on/off switch for tracing |
| `LANGCHAIN_PROJECT` | any name you choose | Optional — defaults to `default` if omitted |

---

### Step 5: Verify — Run the Setup Cell

After saving your `.env`, run the **Section 1 setup cell**. You should see:

```
✅ LangSmith tracing is ENABLED
   Project : claude-langgraph
   API Key : lsv2_pt_xxxx...
```

**Troubleshooting `⚠️ LangSmith tracing is DISABLED`:**
- Ensure `.env` is in the same folder as this notebook
- `LANGCHAIN_TRACING_V2` must be `true` (lowercase, no quotes)
- `LANGSMITH_API_KEY` must be present and non-empty
- Restart the kernel after editing `.env`

---

### LangSmith Pricing

| Plan | Price | Traces/month | Seats |
|------|-------|-------------|-------|
| **Developer** | Free | 5,000 | 1 |
| **Plus** | $39/month | 50,000 | 5 |
| **Enterprise** | Custom | Unlimited | Unlimited |

> The free **Developer** tier is sufficient for this training course.

---

## Section 1 – Install & Setup

```
pip install langsmith langchain-anthropic langgraph python-dotenv
```

### Required `.env` keys

| Variable | Value | Purpose |
|---|---|---|
| `ANTHROPIC_API_KEY` | `sk-ant-...` | Claude API access |
| `LANGSMITH_API_KEY` | `lsv2_...` | LangSmith authentication |
| `LANGCHAIN_TRACING_V2` | `true` | **Enables automatic tracing** |
| `LANGCHAIN_PROJECT` | e.g. `claude-langgraph` | Project name in LangSmith UI |

Setting `LANGCHAIN_TRACING_V2=true` is the **only change** needed to start tracing — every `chat_model.invoke()`, `graph.invoke()`, and `@tool` call is captured automatically.

In [ ]:
%pip install -q langsmith langchain-anthropic langgraph python-dotenv

In [ ]:

# ── STEP 1: Load .env FIRST — before any LangChain imports ───────────────
import os
from dotenv import load_dotenv

load_dotenv(override=True)

# ── STEP 2: Explicitly push LangSmith vars into os.environ ───────────────
# This ensures LangChain picks them up even if already partially imported
langsmith_api_key = os.getenv("LANGSMITH_API_KEY", "")
tracing_v2        = os.getenv("LANGCHAIN_TRACING_V2", "false")
project_name      = os.getenv("LANGCHAIN_PROJECT", "default")

if langsmith_api_key:
    os.environ["LANGSMITH_API_KEY"]       = langsmith_api_key
    os.environ["LANGCHAIN_TRACING_V2"]    = tracing_v2
    os.environ["LANGCHAIN_PROJECT"]       = project_name

# ── STEP 3: Now import LangChain/LangGraph (after env is ready) ───────────
import operator
import math
import datetime
from typing import TypedDict, Annotated, Literal

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage, BaseMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langsmith import Client
from langsmith import traceable

# ── STEP 4: Validate Anthropic key ───────────────────────────────────────
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
if not anthropic_api_key:
    raise ValueError("ANTHROPIC_API_KEY not found. Add it to your .env file.")

# ── STEP 5: Print status ──────────────────────────────────────────────────
tracing_enabled = tracing_v2.lower() == "true"

print("=" * 55)
print("  ENVIRONMENT STATUS")
print("=" * 55)
print(f"  ANTHROPIC_API_KEY      : {'✅ set (' + anthropic_api_key[:12] + '...)' if anthropic_api_key else '❌ MISSING'}")
print(f"  LANGSMITH_API_KEY      : {'✅ set (' + langsmith_api_key[:12] + '...)' if langsmith_api_key else '❌ MISSING'}")
print(f"  LANGCHAIN_TRACING_V2   : {tracing_v2!r}  {'✅' if tracing_enabled else '❌ (must be true without quotes)'}")
print(f"  LANGCHAIN_PROJECT      : {project_name!r}")
print("=" * 55)

if tracing_enabled and langsmith_api_key:
    print("\n✅ LangSmith tracing is ENABLED")
    print(f"   Traces will appear in project: {project_name!r}")
    print("   → Open https://smith.langchain.com → Projects → " + project_name)
else:
    print("\n❌ LangSmith tracing is DISABLED — check the values above")

# ── STEP 6: Create Claude model & LangSmith client ───────────────────────
chat_model = ChatAnthropic(
    model="claude-opus-4-6",
    temperature=0.3,
    anthropic_api_key=anthropic_api_key,
)

ls_client = Client(api_key=langsmith_api_key) if langsmith_api_key else None

from importlib.metadata import version
print(f"\n   LangGraph : {version('langgraph')}")
print(f"   LangSmith : {version('langsmith')}")
print(f"   Model     : claude-opus-4-6")


In [ ]:
#clean code with all the steps and comments
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Literal
import operator
import math
import datetime

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage, BaseMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langsmith import Client, traceable

# Load env
load_dotenv(override=True)

# Read env vars
langsmith_api_key = os.getenv("LANGSMITH_API_KEY", "")
tracing_v2 = os.getenv("LANGCHAIN_TRACING_V2", "false")
project_name = os.getenv("LANGCHAIN_PROJECT", "default")
#anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
# Load ANTHROPIC_API_KEY from .env
anthropic_api_key = os.getenv("LLMGW_API_KEY")
if not anthropic_api_key:
    raise ValueError("LLMGW_API_KEY not found. Add it to your .env file.")

# Push env vars explicitly
os.environ["LANGCHAIN_TRACING_V2"] = tracing_v2
os.environ["LANGCHAIN_PROJECT"] = project_name
if langsmith_api_key:
    os.environ["LANGSMITH_API_KEY"] = langsmith_api_key

# Validate required Anthropic key
if not anthropic_api_key:
    raise ValueError("ANTHROPIC_API_KEY not found. Add it to your .env file.")

# Create model and LangSmith client
chat_model = ChatAnthropic(
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    temperature=0.3,
    anthropic_api_key=anthropic_api_key,
base_url="https://llmgw-wp.tekstac.com"
 
)
'''
chat_model = ChatAnthropic(
    model="claude-opus-4-6",
    temperature=0.3,
    anthropic_api_key=anthropic_api_key,
)'''
ls_client = Client(api_key=langsmith_api_key) if langsmith_api_key else None

# Optional single-line status
tracing_enabled = tracing_v2.lower() == "true"
print(f"Tracing={'ON' if tracing_enabled and langsmith_api_key else 'OFF'} | Project={project_name}")

In [ ]:
# ── Quick Connectivity Test — run this to verify traces appear ───────────
# After running this cell, go to LangSmith → Projects → your project
# You should see a run named "connectivity-test" within 5 seconds.

print(" Running connectivity test...")

if not tracing_enabled or not langsmith_api_key:
    print(" Tracing is disabled. Re-run the setup cell after fixing .env")
else:
    # Verify LangSmith API key is valid
    try:
        projects = list(ls_client.list_projects())
        project_names = [p.name for p in projects]
        print(f" LangSmith API key is valid")
        print(f"   Your projects: {project_names}")
        if project_name not in project_names:
            print(f"\n  WARNING: Project '{project_name}' not found in your account!")
            print(f"   Available projects: {project_names}")
            print(f"   Either create it at smith.langchain.com or change LANGCHAIN_PROJECT in .env")
        else:
            print(f"    Project '{project_name}' exists")
    except Exception as e:
        print(f" LangSmith API key error: {e}")
        print("   Check that LANGSMITH_API_KEY is correct in your .env file")

    # Send a test trace
    print("\n Sending a test Claude call (should appear in LangSmith in ~5s)...")
    try:
        test_response = chat_model.invoke(
            [HumanMessage(content="Say exactly: LangSmith trace test successful")],
            config={"run_name": "connectivity-test", "tags": ["test"]},
        )
        print(f" Claude responded: {test_response.content.strip()}")
        print(f"\n Now check: smith.langchain.com → Projects → {project_name} → Runs")
        print(f"   Look for a run named 'connectivity-test'")
    except Exception as e:
        print(f" Claude call failed: {e}")

In [ ]:
# ── Auto-traced Claude call — no code change needed ──────────────────────
# With LANGCHAIN_TRACING_V2=true this call appears in LangSmith automatically.
print("Section 3  Auto-traced Claude call")
print("=" * 50)

response = chat_model.invoke([
    SystemMessage(content="You are a helpful assistant. Be concise."),
    HumanMessage(content="What are the three pillars of MLOps?"),
])

print("Question : What are the three pillars of MLOps?")
print("\nClaude   :", response.content.strip())

# Token usage is available in the response metadata
usage = response.usage_metadata if hasattr(response, "usage_metadata") else {}
print(f"\nToken usage → input: {usage.get('input_tokens','?')}  "
      f"output: {usage.get('output_tokens','?')}")
print("\n Open smith.langchain.com → your project to see this run traced.")

---

## Section 2 – What is LangSmith Observability?

LangSmith captures the full execution of every AI call as a **trace tree**:

```
Trace (top-level run)
  ├── LangGraph: graph.invoke()
  │     ├── Node: agent_llm_node
  │     │     └── ChatAnthropic.invoke()   ← LLM call (input, output, tokens, latency)
  │     ├── Node: tools_exec_node
  │     │     ├── Tool: add_numbers        ← tool call (args, result)
  │     │     └── Tool: calculate_square_root
  │     └── Node: agent_llm_node  (2nd loop)
  │           └── ChatAnthropic.invoke()
  └── ...
```

### What LangSmith records for each run

| Field | Description |
|---|---|
| **Inputs** | Messages / state sent to the node |
| **Outputs** | Messages / state returned |
| **Latency** | Wall-clock time per node and total |
| **Token usage** | Prompt + completion tokens, cost estimate |
| **Errors** | Stack traces if a node raises an exception |
| **Feedback** | Scores / labels added by humans or eval pipelines |

### Three ways to trace

| Method | Code change? | Use case |
|---|---|---|
| **Auto-trace** (env var) | None — just set `LANGCHAIN_TRACING_V2=true` | All LangChain/LangGraph calls |
| **`@traceable`** | Add decorator | Pure-Python functions outside LangChain |
| **`ls_client.create_run()`** | Manual SDK | Fine-grained control, async pipelines |

In [ ]:
# ── Quick Connectivity Test — run this to verify traces appear ───────────
# After running this cell, go to LangSmith → Projects → your project
# You should see a run named "connectivity-test" within 5 seconds.

print("🔍 Running connectivity test...")

if not tracing_enabled or not langsmith_api_key:
    print("❌ Tracing is disabled. Re-run the setup cell after fixing .env")
else:
    # Verify LangSmith API key is valid
    try:
        projects = list(ls_client.list_projects())
        project_names = [p.name for p in projects]
        print(f"✅ LangSmith API key is valid")
        print(f"   Your projects: {project_names}")
        if project_name not in project_names:
            print(f"\n⚠️  WARNING: Project '{project_name}' not found in your account!")
            print(f"   Available projects: {project_names}")
            print(f"   Either create it at smith.langchain.com or change LANGCHAIN_PROJECT in .env")
        else:
            print(f"   ✅ Project '{project_name}' exists")
    except Exception as e:
        print(f"❌ LangSmith API key error: {e}")
        print("   Check that LANGSMITH_API_KEY is correct in your .env file")

    # Send a test trace
    print("\n📡 Sending a test Claude call (should appear in LangSmith in ~5s)...")
    try:
        test_response = chat_model.invoke(
            [HumanMessage(content="Say exactly: LangSmith trace test successful")],
            config={"run_name": "connectivity-test", "tags": ["test"]},
        )
        print(f"✅ Claude responded: {test_response.content.strip()}")
        print(f"\n👉 Now check: smith.langchain.com → Projects → {project_name} → Runs")
        print(f"   Look for a run named 'connectivity-test'")
    except Exception as e:
        print(f"❌ Claude call failed: {e}")

---

## Section 3 – Automatic Tracing: Zero-Code Observability

With `LANGCHAIN_TRACING_V2=true` in your `.env`, **every** `chat_model.invoke()` is automatically captured. No code changes required.

In [ ]:
# ── Auto-traced Claude call — no code change needed ──────────────────────
# With LANGCHAIN_TRACING_V2=true this call appears in LangSmith automatically.
print("Section 3 – Auto-traced Claude call")
print("=" * 50)

response = chat_model.invoke([
    SystemMessage(content="You are a helpful assistant. Be concise."),
    HumanMessage(content="What are the three pillars of MLOps?"),
])

print("Question : What are the three pillars of MLOps?")
print("\nClaude   :", response.content.strip())

# Token usage is available in the response metadata
usage = response.usage_metadata if hasattr(response, "usage_metadata") else {}
print(f"\nToken usage → input: {usage.get('input_tokens','?')}  "
      f"output: {usage.get('output_tokens','?')}")
print("\n💡 Open smith.langchain.com → your project to see this run traced.")

---

## Section 4 – `@traceable`: Custom Spans for Pure-Python Code

The `@traceable` decorator wraps **any Python function** as a named span in the trace tree — including preprocessing, postprocessing, scoring, or business logic that doesn't call LangChain directly.

```python
from langsmith import traceable

@traceable(name="my_custom_step", run_type="chain")
def my_function(input_data):
    ...
    return result
```

`run_type` options: `"chain"`, `"llm"`, `"tool"`, `"retriever"`, `"embedding"`

In [ ]:
# ── @traceable: custom spans around pure-Python logic ────────────────────
print("Section 4  @traceable Custom Spans")
print("=" * 50)

@traceable(name="preprocess_query", run_type="chain")
def preprocess_query(raw_query: str) -> str:
    """Clean and normalise user input before sending to Claude."""
    cleaned = raw_query.strip().lower()
    # Remove filler words
    for filler in ["please", "can you", "could you", "tell me"]:
        cleaned = cleaned.replace(filler, "").strip()
    return cleaned.capitalize()

@traceable(name="postprocess_answer", run_type="chain")
def postprocess_answer(raw_answer: str, max_chars: int = 300) -> str:
    """Truncate and clean the model's answer for display."""
    answer = raw_answer.strip()
    if len(answer) > max_chars:
        answer = answer[:max_chars].rsplit(" ", 1)[0] + "..."
    return answer

@traceable(name="full_qa_pipeline", run_type="chain")
def full_qa_pipeline(user_input: str) -> str:
    """
    Full pipeline captured as ONE parent trace with two child spans:
      preprocess_query → Claude → postprocess_answer
    """
    # Child span 1: preprocessing
    clean_query = preprocess_query(user_input)
    print(f"  Preprocessed : '{clean_query}'")

    # Child span 2: Claude call (auto-traced by LangChain)
    response = chat_model.invoke([HumanMessage(content=clean_query)])

    # Child span 3: postprocessing
    final_answer = postprocess_answer(response.content)
    return final_answer

# ── Run the pipeline ──────────────────────────────────────────────────────
test_inputs = [
    "Please tell me what is the difference between AI and ML?",
    "Can you explain what LangGraph does?",
]

for q in test_inputs:
    print(f"\n Input  : {q}")
    answer = full_qa_pipeline(q)
    print(f" Answer : {answer}")

print("\n In LangSmith: full_qa_pipeline appears as the parent run.")
print("   Expand it to see preprocess_query → ChatAnthropic → postprocess_answer.")

In [ ]:
# ── @traceable: custom spans around pure-Python logic ────────────────────
print("Section 4 – @traceable Custom Spans")
print("=" * 50)

@traceable(name="preprocess_query", run_type="chain")
def preprocess_query(raw_query: str) -> str:
    """Clean and normalise user input before sending to Claude."""
    cleaned = raw_query.strip().lower()
    # Remove filler words
    for filler in ["please", "can you", "could you", "tell me"]:
        cleaned = cleaned.replace(filler, "").strip()
    return cleaned.capitalize()

@traceable(name="postprocess_answer", run_type="chain")
def postprocess_answer(raw_answer: str, max_chars: int = 300) -> str:
    """Truncate and clean the model's answer for display."""
    answer = raw_answer.strip()
    if len(answer) > max_chars:
        answer = answer[:max_chars].rsplit(" ", 1)[0] + "..."
    return answer

@traceable(name="full_qa_pipeline", run_type="chain")
def full_qa_pipeline(user_input: str) -> str:
    """
    Full pipeline captured as ONE parent trace with two child spans:
      preprocess_query → Claude → postprocess_answer
    """
    # Child span 1: preprocessing
    clean_query = preprocess_query(user_input)
    print(f"  Preprocessed : '{clean_query}'")

    # Child span 2: Claude call (auto-traced by LangChain)
    response = chat_model.invoke([HumanMessage(content=clean_query)])

    # Child span 3: postprocessing
    final_answer = postprocess_answer(response.content)
    return final_answer

# ── Run the pipeline ──────────────────────────────────────────────────────
test_inputs = [
    "Please tell me what is the difference between AI and ML?",
    "Can you explain what LangGraph does?",
]

for q in test_inputs:
    print(f"\n❓ Input  : {q}")
    answer = full_qa_pipeline(q)
    print(f"✅ Answer : {answer}")

print("\n💡 In LangSmith: full_qa_pipeline appears as the parent run.")
print("   Expand it to see preprocess_query → ChatAnthropic → postprocess_answer.")

---

## Section 5 – Tracing the Cyclic Agent Graph (Loops)

The **cyclic agent graph** from Part B (`agent ↔ tools` loop) is traced entirely automatically. LangSmith shows each loop iteration as a separate child run under the parent `graph.invoke()` trace — making it easy to see how many tool calls were made and how long each took.

```
Trace: graph.invoke()
  ├── Iteration 1 – agent node  →  ChatAnthropic (proposes tool call)
  ├── Iteration 1 – tools node  →  calculate_square_root(225)
  └── Iteration 2 – agent node  →  ChatAnthropic (final answer, no tool calls)
```

In [ ]:
# ── Cyclic Agent Graph — same as Part B, traced automatically ────────────
print("Section 5 – Cyclic Agent Graph (traced)")
print("=" * 50)

@tool
def add_numbers(a: float, b: float) -> float:
    """Add two numbers together and return the result."""
    return a + b

@tool
def get_current_date() -> str:
    """Return today's date in YYYY-MM-DD format."""
    return datetime.date.today().isoformat()

@tool
def calculate_square_root(number: float) -> float:
    """Calculate the square root of a non-negative number."""
    if number < 0:
        return "Error: cannot compute square root of a negative number."
    return round(math.sqrt(number), 4)

agent_tools    = [add_numbers, get_current_date, calculate_square_root]
tool_map       = {t.name: t for t in agent_tools}
llm_with_tools = chat_model.bind_tools(agent_tools)

class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]

def agent_llm_node(state: AgentState) -> dict:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

def tools_exec_node(state: AgentState) -> dict:
    last_msg = state["messages"][-1]
    results  = []
    for tc in last_msg.tool_calls:
        output = tool_map[tc["name"]].invoke(tc["args"])
        results.append(ToolMessage(content=str(output), tool_call_id=tc["id"]))
    return {"messages": results}

def should_continue(state: AgentState) -> Literal["tools", "__end__"]:
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return END

agent_builder = StateGraph(AgentState)
agent_builder.add_node("agent", agent_llm_node)
agent_builder.add_node("tools", tools_exec_node)
agent_builder.add_edge(START, "agent")
agent_builder.add_conditional_edges("agent", should_continue)
agent_builder.add_edge("tools", "agent")   # cycle

agent_graph = agent_builder.compile()

# ── Run queries — each becomes a separate top-level trace in LangSmith ──
queries = [
    "What is 47 plus 89?",
    "What is today's date and the square root of 225?",
]

for q in queries:
    print(f"\n❓ Query : {q}")
    r = agent_graph.invoke({"messages": [HumanMessage(content=q)]})
    print(f"✅ Answer: {r['messages'][-1].content.strip()}")
    print(f"   Trace : {len(r['messages'])} messages exchanged")

print("\n💡 LangSmith shows each loop iteration as a separate child run.")
print("   Multi-tool queries will show 2 Claude nodes + 1 tools node.")

---

## Section 6 – Tracing Memory / MemorySaver Graph

When a `MemorySaver` graph is used across multiple turns, **each `.invoke()` call is a separate trace** in LangSmith. You can use the `thread_id` as a filter tag to group all turns of one session together.

> **Tip:** Set `run_name` in the config to give each trace a meaningful label in the dashboard:
> ```python
> config = {"configurable": {"thread_id": "alice-001"}, "run_name": "alice-turn-1"}
> ```

In [ ]:
# ── Memory Graph with run_name tags for LangSmith grouping ───────────────
print("Section 6 – Memory Graph (traced across turns)")
print("=" * 50)

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]

def chat_node(state: ChatState) -> dict:
    return {"messages": [chat_model.invoke(state["messages"])]}

chat_builder = StateGraph(ChatState)
chat_builder.add_node("chat", chat_node)
chat_builder.add_edge(START, "chat")
chat_builder.add_edge("chat", END)

mem_saver       = MemorySaver()
persistent_chat = chat_builder.compile(checkpointer=mem_saver)

alice_turns = [
    ("Hi! My name is Alice. I'm a cloud architect at a fintech firm.",   "alice-turn-1"),
    ("I mainly use AWS and Kubernetes day to day.",                       "alice-turn-2"),
    ("What do you know about me so far?",                                  "alice-turn-3"),
]

print("── Alice's session (thread_id = alice-001) ──")
for msg, run_name in alice_turns:
    cfg = {
        "configurable": {"thread_id": "alice-001"},
        "run_name": run_name,   # ← visible as trace name in LangSmith
    }
    r     = persistent_chat.invoke({"messages": [HumanMessage(content=msg)]}, config=cfg)
    reply = r["messages"][-1].content.strip()
    print(f"\n👤 Alice : {msg}")
    print(f"🤖 AI    : {reply[:200]}")

print("\n💡 In LangSmith filter by metadata 'thread_id=alice-001' to see all 3 turns together.")
print("   Each turn shows its own token count and latency.")

---

## Section 7 – Tracing HITL (Human-in-the-Loop)

HITL graphs produce **two separate trace entries** per interaction:
1. **First `.invoke()`** — captures the pause point; the trace shows the interrupted state
2. **Resume `.invoke(None, ...)`** — a continuation trace linked to the same `thread_id`

This lets you see in LangSmith exactly where a human approved or rejected a tool call and what the final outcome was.

In [ ]:
# ── HITL Graph — two-phase trace (pause + resume) ────────────────────────
print("Section 7 – HITL Tracing (interrupt_before)")
print("=" * 50)

hitl_ckpt  = MemorySaver()
hitl_graph = agent_builder.compile(
    checkpointer=hitl_ckpt,
    interrupt_before=["tools"],
)

# ── Demo 1: APPROVE ───────────────────────────────────────────────────────
cfg_approve = {
    "configurable": {"thread_id": "hitl-trace-approve"},
    "run_name": "hitl-phase1-propose",   # Phase 1 trace label in LangSmith
}
query = "What is the square root of 625?"

print(f"\n❓ Query : {query}")
paused = hitl_graph.invoke(
    {"messages": [HumanMessage(content=query)]},
    config=cfg_approve,
)

last_msg = paused["messages"][-1]
print("\n⏸️  PAUSED — proposed tool calls:")
if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
    for tc in last_msg.tool_calls:
        print(f"   • {tc['name']}({tc['args']})")

# Resume (Phase 2 trace)
cfg_approve["run_name"] = "hitl-phase2-resume"
print("\n✅ Human: APPROVED — resuming...")
final = hitl_graph.invoke(None, config=cfg_approve)
print(f"🤖 Final : {final['messages'][-1].content.strip()}")

# ── Demo 2: REJECT ────────────────────────────────────────────────────────
cfg_reject = {
    "configurable": {"thread_id": "hitl-trace-reject"},
    "run_name": "hitl-phase1-rejected",
}
query2 = "What is 100 plus 200?"

print(f"\n❓ Query : {query2}")
paused2 = hitl_graph.invoke(
    {"messages": [HumanMessage(content=query2)]},
    config=cfg_reject,
)
print("❌ Human: REJECTED — not resuming")
print("   (Phase 1 trace is recorded; Phase 2 trace never created)")

print("\n💡 LangSmith shows hitl-phase1-propose and hitl-phase2-resume as linked traces.")
print("   The rejected run only has Phase 1 — clear audit trail.")

---

## Section 8 – Tracing the Loan Approval System

The Loan Approval System from Part B is a real-world HITL workflow. With LangSmith you can monitor:
- Which verification tools were called for each applicant
- How long the credit/income/DTI checks took
- What recommendation Claude produced
- Whether the loan officer approved or rejected
- Full token cost per application

In [ ]:
# ── Loan Approval System — traced with run_name + metadata tags ──────────
print("Section 8 – Loan Approval System (fully traced)")
print("=" * 60)

# ── Loan tools ────────────────────────────────────────────────────────────
@tool
def check_credit_score(applicant_id: str) -> str:
    """Check the applicant's credit score."""
    scores = {
        "APP-001": {"score": 750, "status": "Excellent"},
        "APP-002": {"score": 620, "status": "Fair"},
        "APP-003": {"score": 800, "status": "Excellent"},
    }
    data = scores.get(applicant_id, {"score": 700, "status": "Good"})
    return f"Credit Score: {data['score']} ({data['status']})"

@tool
def verify_income(applicant_id: str) -> str:
    """Verify annual income."""
    incomes = {"APP-001": 85000, "APP-002": 45000, "APP-003": 120000}
    income = incomes.get(applicant_id, 70000)
    return f"Annual Income: ${income:,}"

@tool
def calculate_dti(applicant_id: str, loan_amount: float) -> str:
    """Calculate Debt-to-Income ratio."""
    incomes = {"APP-001": 85000, "APP-002": 45000, "APP-003": 120000}
    annual_income   = incomes.get(applicant_id, 70000)
    monthly_payment = (loan_amount / 60) + 500
    monthly_income  = annual_income / 12
    dti_ratio       = (monthly_payment / monthly_income) * 100
    status          = "✅ PASS" if dti_ratio < 43 else "❌ FAIL"
    return f"Debt-to-Income Ratio: {dti_ratio:.1f}% {status} (max: 43%)"

@tool
def verify_employment(applicant_id: str) -> str:
    """Verify employment status."""
    employments = {
        "APP-001": "Employed - Tech Corp (5 years)",
        "APP-002": "Self-employed - Freelancer (2 years)",
        "APP-003": "Employed - Finance Co (8 years)",
    }
    return employments.get(applicant_id, "Employed (3 years)")

# ── Agent setup ───────────────────────────────────────────────────────────
loan_tools     = [check_credit_score, verify_income, calculate_dti, verify_employment]
tool_map_loans = {t.name: t for t in loan_tools}
llm_loans      = chat_model.bind_tools(loan_tools)

class LoanState(TypedDict):
    applicant_id: str
    loan_amount:  float
    messages:     Annotated[list[BaseMessage], operator.add]

def loan_agent(state: LoanState) -> dict:
    system_prompt = f"""You are a loan approval specialist. Analyze this loan application:
    - Applicant: {state['applicant_id']}
    - Loan Amount: ${state['loan_amount']:,.0f}

    Use ALL available tools to verify credit score, income, DTI ratio, and employment.
    Then provide a CLEAR RECOMMENDATION: APPROVE or REJECT with reasoning."""

    return {"messages": [llm_loans.invoke(
        [SystemMessage(content=system_prompt)] + state["messages"]
    )]}

def loan_tools_exec(state: LoanState) -> dict:
    last_msg = state["messages"][-1]
    results  = []
    for tc in last_msg.tool_calls:
        output = tool_map_loans[tc["name"]].invoke(tc["args"])
        results.append(ToolMessage(content=output, tool_call_id=tc["id"]))
    return {"messages": results}

def should_continue_loan(state: LoanState) -> Literal["loan_tools", "__end__"]:
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "loan_tools"
    return END

loan_builder = StateGraph(LoanState)
loan_builder.add_node("loan_agent", loan_agent)
loan_builder.add_node("loan_tools", loan_tools_exec)
loan_builder.add_edge(START, "loan_agent")
loan_builder.add_conditional_edges("loan_agent", should_continue_loan)
loan_builder.add_edge("loan_tools", "loan_agent")

loan_hitl = loan_builder.compile(
    checkpointer=MemorySaver(),
    interrupt_before=["loan_tools"],
)

def get_final_answer(result):
    for msg in reversed(result["messages"]):
        if hasattr(msg, "content") and isinstance(msg.content, str) and msg.content.strip():
            return msg.content.strip()
    return "No recommendation found"

# ── Non-interactive version (auto-approves) for tracing demo ─────────────
def process_loan_traced(applicant_id: str, loan_amount: float, auto_approve: bool = True):
    """
    Process loan application with LangSmith tracing.
    run_name tags each phase so they're easy to find in the dashboard.
    """
    thread_id = f"loan-trace-{applicant_id}"

    print(f"\n{'─'*60}")
    print(f"📋 Applicant : {applicant_id}")
    print(f"💰 Loan      : ${loan_amount:,.0f}")
    print(f"{'─'*60}")

    # Phase 1 — agent proposes tool calls
    cfg_phase1 = {
        "configurable": {"thread_id": thread_id},
        "run_name": f"loan-{applicant_id}-phase1-propose",
        "tags": ["loan-approval", applicant_id],           # searchable in LangSmith
        "metadata": {"applicant_id": applicant_id, "loan_amount": loan_amount},
    }
    paused = loan_hitl.invoke(
        {
            "applicant_id": applicant_id,
            "loan_amount":  loan_amount,
            "messages": [HumanMessage(content="Please analyze this loan application.")],
        },
        config=cfg_phase1,
    )

    last_msg = paused["messages"][-1]
    print("⏸️  PAUSED — will run:")
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        for tc in last_msg.tool_calls:
            print(f"   • {tc['name']}")

    decision = "approved" if auto_approve else input("\n👤 Decision (approved/rejected): ").strip().lower()
    print(f"\n{'✅' if decision == 'approved' else '❌'} Decision: {decision.upper()}")

    if decision != "approved":
        print("   Application stopped.")
        return

    # Phase 2 — resume and get recommendation
    cfg_phase2 = {
        "configurable": {"thread_id": thread_id},
        "run_name": f"loan-{applicant_id}-phase2-recommend",
        "tags": ["loan-approval", applicant_id],
        "metadata": {"applicant_id": applicant_id, "loan_amount": loan_amount, "officer_decision": "approved"},
    }
    final          = loan_hitl.invoke(None, config=cfg_phase2)
    recommendation = get_final_answer(final)

    print(f"\n🤖 Recommendation:\n{recommendation[:400]}")
    print("─" * 60)

# ── Run two applications ──────────────────────────────────────────────────
process_loan_traced("APP-001", 250000, auto_approve=True)
process_loan_traced("APP-002", 150000, auto_approve=True)

print("\n💡 In LangSmith: filter by tag 'loan-approval' to see all loan runs.")
print("   Each run has metadata (applicant_id, loan_amount) for filtering/export.")

---

## Section 9 – Adding Feedback & Evaluation Scores

**Feedback** lets you attach a human or automated score to any traced run. This is how you build an evaluation dataset over time.

```python
ls_client.create_feedback(
    run_id=run_id,
    key="correctness",      # metric name — anything you define
    score=1.0,              # 0.0 = bad, 1.0 = good
    comment="Answer is factually correct and concise",
)
```

Common feedback keys:
| Key | Type | Description |
|---|---|---|
| `correctness` | 0–1 | Is the answer factually right? |
| `relevance` | 0–1 | Does it address the question? |
| `tool_selection` | 0–1 | Did the agent pick the right tools? |
| `approved` | bool | Did a human approve this action? |

In [ ]:
# ── Attaching Feedback to a traced run ───────────────────────────────────
print("Section 9 – Feedback & Evaluation Scores")
print("=" * 50)

if not ls_client:
    print("⚠️  LangSmith client not available (LANGSMITH_API_KEY missing).")
    print("   Set LANGSMITH_API_KEY in .env and re-run.")
else:
    # Run a traced call and capture the run_id
    import uuid
    from langsmith.run_helpers import get_current_run_tree

    @traceable(name="qa_with_feedback", run_type="chain")
    def qa_with_feedback(question: str) -> dict:
        """Ask Claude and return both the answer and the current run_id."""
        response = chat_model.invoke([HumanMessage(content=question)])
        answer   = response.content.strip()

        # Capture the run_id of THIS traceable span
        run_tree = get_current_run_tree()
        run_id   = str(run_tree.id) if run_tree else None

        return {"answer": answer, "run_id": run_id}

    question = "What is the capital of France?"
    result   = qa_with_feedback(question)

    print(f"❓ Question : {question}")
    print(f"✅ Answer   : {result['answer']}")
    print(f"🔖 Run ID   : {result['run_id']}")

    # Attach human feedback to this run
    if result["run_id"]:
        ls_client.create_feedback(
            run_id  = result["run_id"],
            key     = "correctness",
            score   = 1.0,
            comment = "Answer is correct — Paris is the capital of France.",
        )
        ls_client.create_feedback(
            run_id  = result["run_id"],
            key     = "relevance",
            score   = 1.0,
            comment = "Response directly addresses the question.",
        )
        print(f"\n📊 Feedback submitted for run {result['run_id']}")
        print("   correctness = 1.0  |  relevance = 1.0")
        print("   Open LangSmith → Runs → click this run → Feedback tab to verify.")

---

## Section 10 – LangSmith Dashboard: What to Look For

### Navigating the UI

```
smith.langchain.com
  └── Projects
        └── claude-langgraph       ← your LANGCHAIN_PROJECT name
              ├── Runs             ← every traced call appears here
              ├── Datasets         ← saved examples for evaluation
              └── Experiments      ← batch eval results
```

### Key panels per Run

| Panel | What it shows |
|---|---|
| **Input / Output** | Exact messages sent to and received from Claude |
| **Timeline** | Waterfall chart of node execution times |
| **Tokens** | Prompt tokens, completion tokens, estimated cost |
| **Feedback** | All scores attached to this run |
| **Metadata** | Custom tags and key-value pairs you set |
| **Child Runs** | Nested spans (tool calls, LangGraph nodes) |

### Useful filters in the Runs view

| Filter | Example | Use case |
|---|---|---|
| `run_name` | `loan-APP-001-*` | Find all runs for one applicant |
| `tags` | `loan-approval` | All loan approval runs |
| `metadata` | `applicant_id=APP-002` | Single applicant history |
| `feedback.correctness > 0.8` | | Only high-quality runs |
| `error is not null` | | Find all failed runs |

### LangSmith Observability — Summary

| Feature | Benefit |
|---|---|
| **Zero-code tracing** | `LANGCHAIN_TRACING_V2=true` captures everything |
| **`@traceable`** | Adds custom spans for non-LangChain Python code |
| **`run_name` / `tags` / `metadata`** | Organise and search thousands of runs |
| **Token tracking** | Monitor cost per workflow or per user |
| **Feedback scores** | Build evaluation datasets from production traffic |
| **HITL audit trail** | Complete record of approve/reject decisions |
| **Error visibility** | Stack traces surfaced without digging through logs |

---

## Section 11 – Creating & Managing Datasets

A **Dataset** in LangSmith is a collection of (input, expected_output) example pairs. You use datasets to:
- Run **regression tests** when you change your prompt or model
- Track quality over time with **Experiments**
- Build a **golden set** of verified answers

### Two ways to create a dataset

| Method | When to use |
|--------|------------|
| **Manually** via SDK | You know the examples upfront |
| **From production runs** | Save real user runs as examples |

```python
# Create dataset manually
dataset = ls_client.create_dataset("my-qa-dataset", description="QA pairs for evaluation")

# Add examples
ls_client.create_examples(
    inputs=[{"question": "What is the capital of France?"}],
    outputs=[{"answer": "Paris"}],
    dataset_id=dataset.id,
)
```

In [ ]:
# ── Section 11: Creating & Managing Datasets ────────────────────────────
print("Section 11 – LangSmith Datasets")
print("=" * 55)

if not ls_client:
    print("⚠️  LangSmith client not available (LANGSMITH_API_KEY missing).")
else:
    DATASET_NAME = "claude-qa-training-dataset"

    # ── Step 1: Create (or reuse) a dataset ──────────────────────────────
    existing = [d for d in ls_client.list_datasets() if d.name == DATASET_NAME]
    if existing:
        dataset = existing[0]
        print(f"✅ Reusing existing dataset: '{DATASET_NAME}'  (id: {dataset.id})")
    else:
        dataset = ls_client.create_dataset(
            DATASET_NAME,
            description="QA pairs for Claude LangGraph training — evaluating factual accuracy",
        )
        print(f"✅ Created new dataset: '{DATASET_NAME}'  (id: {dataset.id})")

    # ── Step 2: Add example (input / expected output) pairs ──────────────
    examples = [
        {"input": "What is the capital of France?",           "output": "Paris"},
        {"input": "What does LLM stand for?",                  "output": "Large Language Model"},
        {"input": "Name the three pillars of MLOps.",          "output": "People, Processes, Technology"},
        {"input": "What is LangGraph used for?",               "output": "Building stateful, cyclic LLM agent workflows"},
        {"input": "What company makes the Claude AI model?",   "output": "Anthropic"},
    ]

    ls_client.create_examples(
        inputs  = [{"question": e["input"]}  for e in examples],
        outputs = [{"answer":   e["output"]} for e in examples],
        dataset_id = dataset.id,
    )
    print(f"\n📦 Added {len(examples)} examples to dataset.")
    print("\n   Examples added:")
    for i, e in enumerate(examples, 1):
        print(f"   {i}. Q: {e['input']}")
        print(f"      A: {e['output']}")

    # ── Step 3: List all datasets in your project ─────────────────────────
    print(f"\n📂 All datasets in your LangSmith workspace:")
    for d in ls_client.list_datasets():
        count = sum(1 for _ in ls_client.list_examples(dataset_id=d.id))
        print(f"   • {d.name}  ({count} examples)")

    print(f"\n💡 Open LangSmith → Datasets → '{DATASET_NAME}' to see all examples.")
    print("   You can edit, delete, or add more examples directly in the UI.")

---

## Section 12 – Automated Evaluation with Custom Evaluators

**Evaluation** = running every example in a dataset through your app and scoring each output automatically.

### How it works

```
Dataset (5 examples)
    ↓  run each through your app
Experiment (5 runs)
    ↓  score each output with your evaluator function
Results
    ├── example 1: correctness = 1.0  ✅
    ├── example 2: correctness = 0.0  ❌
    └── ...
```

### The evaluator function signature

```python
def my_evaluator(run, example) -> dict:
    """
    run     = the actual app run (has run.outputs["answer"])
    example = the dataset example (has example.outputs["answer"])
    """
    score = 1.0 if answer_correct else 0.0
    return {"key": "correctness", "score": score, "comment": "..."}
```

LangSmith calls this function for **every example** in the dataset and aggregates the scores into an Experiment.

In [ ]:
# ── Section 12: Automated Evaluation with Custom Evaluators ─────────────
print("Section 12 – Automated Evaluation")
print("=" * 55)

if not ls_client:
    print("⚠️  LangSmith client not available (LANGSMITH_API_KEY missing).")
else:
    from langsmith.evaluation import evaluate

    # ── The app we want to evaluate ──────────────────────────────────────
    # This function is called once per dataset example
    @traceable(name="eval_qa_app", run_type="chain")
    def qa_app_under_test(inputs: dict) -> dict:
        """Our LangGraph-backed QA app — this is what the evaluator will test."""
        question = inputs.get("question", "")
        response = chat_model.invoke([
            SystemMessage(content="Answer the question concisely in 1 sentence."),
            HumanMessage(content=question),
        ])
        return {"answer": response.content.strip()}

    # ── Evaluator 1: Exact keyword match ─────────────────────────────────
    def keyword_match_evaluator(run, example) -> dict:
        """
        Check if the expected answer keywords appear in Claude's response.
        Simple but effective for factual QA.
        """
        actual   = (run.outputs or {}).get("answer", "").lower()
        expected = (example.outputs or {}).get("answer", "").lower()

        # Check if key words from expected answer are present
        expected_words = [w for w in expected.split() if len(w) > 3]
        matches        = sum(1 for w in expected_words if w in actual)
        score          = matches / len(expected_words) if expected_words else 0.0
        passed         = score >= 0.5

        return {
            "key":     "keyword_match",
            "score":   round(score, 2),
            "comment": f"Matched {matches}/{len(expected_words)} key words. "
                       f"Expected keywords: {expected_words}"
        }

    # ── Evaluator 2: Length reasonableness check ─────────────────────────
    def length_evaluator(run, example) -> dict:
        """Penalise answers that are too long (verbose) or too short (incomplete)."""
        answer      = (run.outputs or {}).get("answer", "")
        word_count  = len(answer.split())
        # ideal: 5–50 words for a concise factual answer
        if 5 <= word_count <= 50:
            score, comment = 1.0, f"Good length: {word_count} words"
        elif word_count < 5:
            score, comment = 0.3, f"Too short: {word_count} words"
        else:
            score, comment = 0.6, f"Too verbose: {word_count} words (target ≤ 50)"
        return {"key": "response_length", "score": score, "comment": comment}

    # ── Run the evaluation ────────────────────────────────────────────────
    DATASET_NAME = "claude-qa-training-dataset"
    print(f"🔍 Running evaluation against dataset: '{DATASET_NAME}'")
    print(f"   Evaluators: keyword_match, response_length\n")

    results = evaluate(
        qa_app_under_test,
        data        = DATASET_NAME,
        evaluators  = [keyword_match_evaluator, length_evaluator],
        experiment_prefix = "claude-opus-4-6-eval",
    )

    # ── Print results summary ─────────────────────────────────────────────
    print("\n📊 Evaluation Results:")
    print(f"   Experiment URL: {results.experiment_results_url if hasattr(results, 'experiment_results_url') else 'See LangSmith UI'}")

    keyword_scores = []
    length_scores  = []

    for r in results:
        run     = r.get("run", {})
        evals   = r.get("evaluation_results", {}).get("results", [])
        q = r.get("example", {})
        inputs  = q.inputs if hasattr(q, "inputs") else {}
        print(f"\n   Q: {inputs.get('question', 'N/A')[:55]}")
        print(f"   A: {(run.outputs or {}).get('answer','')[:80]}")
        for ev in evals:
            score = getattr(ev, "score", "?")
            key   = getattr(ev, "key",   "?")
            print(f"   {key}: {score}")
            if key == "keyword_match":
                keyword_scores.append(score)
            elif key == "response_length":
                length_scores.append(score)

    if keyword_scores:
        avg_km = sum(keyword_scores) / len(keyword_scores)
        avg_rl = sum(length_scores) / len(length_scores) if length_scores else 0
        print(f"\n{'─'*55}")
        print(f"   Avg keyword_match   : {avg_km:.2f} / 1.0")
        print(f"   Avg response_length : {avg_rl:.2f} / 1.0")
        print(f"   Total examples run  : {len(keyword_scores)}")

    print("\n💡 Open LangSmith → Experiments tab to see the full comparison table.")
    print("   Run this cell again after changing the model/prompt to compare results.")

---

## Section 13 – Latency & Cost Monitoring

In production you need to track:
- **Latency** — Is my agent getting slower over time? Which node is the bottleneck?
- **Token usage** — How many tokens does each workflow consume? What does it cost?
- **Cost estimation** — Token cost multiplied by model pricing

### Claude Pricing Reference (approximate)

| Model | Input (per 1M tokens) | Output (per 1M tokens) |
|-------|-----------------------|------------------------|
| claude-opus-4-6 | $15 | $75 |
| claude-sonnet-3-7 | $3 | $15 |
| claude-haiku-3-5 | $0.80 | $4 |

> LangSmith tracks token counts per run. Multiply by price to get cost estimates.

### Using the LangSmith SDK to pull run statistics

```python
runs = ls_client.list_runs(project_name="claude-langgraph", limit=100)
for run in runs:
    latency = run.end_time - run.start_time   # timedelta
    tokens  = run.total_tokens
```

In [ ]:
# ── Section 13: Latency & Cost Monitoring ───────────────────────────────
print("Section 13 – Latency & Cost Monitoring")
print("=" * 55)

# ── Step 1: Run 3 Claude calls with different complexity ─────────────────
from time import time

cost_experiments = [
    ("simple",  "What is 2 + 2?"),
    ("medium",  "Explain what LangGraph is in 2-3 sentences."),
    ("complex", "Compare Anthropic Claude, OpenAI GPT-4, and Google Gemini "
                "on safety, reasoning, and coding ability. Use a table."),
]

local_stats = []
print("Running 3 calls with increasing complexity...\n")

for label, q in cost_experiments:
    t_start  = time()
    response = chat_model.invoke([HumanMessage(content=q)])
    t_end    = time()

    latency      = round(t_end - t_start, 2)
    usage        = response.usage_metadata if hasattr(response, "usage_metadata") else {}
    input_tokens = usage.get("input_tokens", 0)
    out_tokens   = usage.get("output_tokens", 0)
    total_tokens = input_tokens + out_tokens

    # Cost estimate (claude-opus-4-6 pricing)
    cost_usd = (input_tokens / 1_000_000 * 15) + (out_tokens / 1_000_000 * 75)

    local_stats.append({
        "label":         label,
        "latency_s":     latency,
        "input_tokens":  input_tokens,
        "output_tokens": out_tokens,
        "total_tokens":  total_tokens,
        "cost_usd":      cost_usd,
    })

    print(f"  [{label.upper()}]")
    print(f"    Q       : {q[:60]}...")
    print(f"    Latency : {latency}s")
    print(f"    Tokens  : {input_tokens} in + {out_tokens} out = {total_tokens} total")
    print(f"    Cost    : ${cost_usd:.6f} USD")
    print()

# ── Step 2: Summary table ─────────────────────────────────────────────────
print("─" * 55)
print(f"{'Query':<10} {'Latency':>9} {'Tokens':>8} {'Est. Cost':>12}")
print("─" * 55)
for s in local_stats:
    print(f"{s['label']:<10} {s['latency_s']:>8.2f}s {s['total_tokens']:>8} "
          f"  ${s['cost_usd']:.6f}")

total_cost = sum(s["cost_usd"] for s in local_stats)
print("─" * 55)
print(f"  TOTAL cost for these 3 calls: ${total_cost:.6f}")

# ── Step 3: Pull recent runs from LangSmith if available ─────────────────
print()
if ls_client:
    print("📈 Pulling last 10 runs from LangSmith for server-side stats...")
    try:
        runs = list(ls_client.list_runs(
            project_name = os.getenv("LANGCHAIN_PROJECT", "claude-langgraph"),
            run_type     = "llm",
            limit        = 10,
        ))
        if runs:
            latencies  = []
            token_totals = []
            for r in runs:
                if r.end_time and r.start_time:
                    latencies.append((r.end_time - r.start_time).total_seconds())
                if hasattr(r, "total_tokens") and r.total_tokens:
                    token_totals.append(r.total_tokens)

            print(f"\n   Runs fetched   : {len(runs)}")
            if latencies:
                print(f"   Avg latency    : {sum(latencies)/len(latencies):.2f}s")
                print(f"   Max latency    : {max(latencies):.2f}s")
                print(f"   Min latency    : {min(latencies):.2f}s")
            if token_totals:
                print(f"   Avg tokens/run : {int(sum(token_totals)/len(token_totals))}")
        else:
            print("   No LLM runs found yet — run more cells first!")
    except Exception as e:
        print(f"   Could not fetch runs: {e}")

print("\n💡 In LangSmith Dashboard → Monitoring tab: view latency/cost trend charts.")
print("   Filter by date range, model, or run_name to isolate bottlenecks.")

---

## Section 14 – Error Tracking & Debugging a Failing Agent

LangSmith captures **errors inside graph nodes automatically**. When a node raises an exception:
- The trace tree shows the exact node that failed
- The error message and full stack trace are stored
- Inputs to the failing node are preserved — so you can reproduce the bug

### The debugging workflow

```
Production Error Report
      ↓
Open LangSmith → Runs → filter by "error is not null"
      ↓
Click the failing run → Timeline view
      ↓
Identify which node failed → inspect its Input
      ↓
Click "Open in Playground" to reproduce + fix
```

### Deliberate error scenarios to understand

| Scenario | What you see in LangSmith |
|----------|--------------------------|
| Node raises `ValueError` | Red node in timeline; full stack trace |
| Tool called with wrong args | Tool span shows error; agent may retry |
| LLM timeout | Latency spike + error on the ChatAnthropic span |
| Missing env variable | Setup cell fails — no trace created |

In [ ]:
# ── Section 14: Error Tracking & Debugging a Failing Agent ──────────────
print("Section 14 – Error Tracking & Debugging")
print("=" * 55)

# ─── Scenario A: A node raises an error intentionally ────────────────────
# LangSmith captures this as a failed run with the full error

class DebugState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    fail_mode: str   # "divide_by_zero" | "bad_tool_arg" | "ok"

@tool
def safe_divide(numerator: float, denominator: float) -> float:
    """Divide numerator by denominator. Raises error if denominator is zero."""
    if denominator == 0:
        raise ValueError("Division by zero is not allowed!")
    return round(numerator / denominator, 4)

debug_tools    = [safe_divide]
tool_map_debug = {t.name: t for t in debug_tools}
llm_debug      = chat_model.bind_tools(debug_tools)

def debug_agent_node(state: DebugState) -> dict:
    """Agent node — will ask to run safe_divide."""
    return {"messages": [llm_debug.invoke(state["messages"])]}

def debug_tools_node(state: DebugState) -> dict:
    """Tool execution node — deliberately passes bad args if fail_mode set."""
    last_msg = state["messages"][-1]
    results  = []
    for tc in last_msg.tool_calls:
        args = tc["args"]

        # Inject bad argument to simulate runtime error
        if state.get("fail_mode") == "divide_by_zero":
            args = {"numerator": args.get("numerator", 10), "denominator": 0}

        try:
            output = tool_map_debug[tc["name"]].invoke(args)
            results.append(ToolMessage(
                content      = str(output),
                tool_call_id = tc["id"],
            ))
        except ValueError as e:
            # Capture the error gracefully into the message stream
            results.append(ToolMessage(
                content      = f"❌ ERROR in {tc['name']}: {str(e)}",
                tool_call_id = tc["id"],
            ))
    return {"messages": results}

def should_continue_debug(state: DebugState) -> str:
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "debug_tools"
    return END

debug_builder = StateGraph(DebugState)
debug_builder.add_node("debug_agent", debug_agent_node)
debug_builder.add_node("debug_tools", debug_tools_node)
debug_builder.add_edge(START, "debug_agent")
debug_builder.add_conditional_edges("debug_agent", should_continue_debug)
debug_builder.add_edge("debug_tools", "debug_agent")
debug_graph = debug_builder.compile()

# ── Run 1: Normal — should succeed ───────────────────────────────────────
print("\n🟢 Run 1: Normal (10 ÷ 2 = 5)")
r1 = debug_graph.invoke({
    "messages":  [HumanMessage(content="What is 10 divided by 2? Use safe_divide.")],
    "fail_mode": "ok",
}, config={"run_name": "debug-run-success"})
print(f"   Result : {r1['messages'][-1].content.strip()[:120]}")

# ── Run 2: Injected divide-by-zero — captured in LangSmith ───────────────
print("\n🔴 Run 2: Deliberate divide-by-zero error")
r2 = debug_graph.invoke({
    "messages":  [HumanMessage(content="What is 10 divided by 5? Use safe_divide.")],
    "fail_mode": "divide_by_zero",
}, config={"run_name": "debug-run-error", "tags": ["error-demo"]})
print(f"   Result : {r2['messages'][-1].content.strip()[:200]}")

# ── Step 3: Query failed runs from LangSmith ─────────────────────────────
print("\n📋 Querying LangSmith for runs tagged 'error-demo'...")
if ls_client:
    try:
        error_runs = list(ls_client.list_runs(
            project_name = os.getenv("LANGCHAIN_PROJECT", "claude-langgraph"),
            filter       = 'has(tags, "error-demo")',
            limit        = 5,
        ))
        print(f"   Found {len(error_runs)} run(s) tagged 'error-demo'")
        for r in error_runs:
            print(f"   • {r.name}  |  status: {r.status}  |  id: {str(r.id)[:12]}...")
    except Exception as e:
        print(f"   Could not query: {e}")
else:
    print("   LangSmith client not available — run skipped.")

print("\n💡 In LangSmith → Runs → add filter 'Error is not null'")
print("   Click on 'debug-run-error' → Timeline → see which node turned red.")
print("   Click that node → view the exact input args that caused the failure.")

---

## Section 15 – Summary, Checklist & Best Practices

### Complete LangSmith Feature Coverage — 2-Hour Session Recap

| Section | Topic | Key Takeaway |
|---------|-------|-------------|
| Theory | Why Observability? | AI apps are black boxes without tracing |
| Setup | Account + API Key | 5-minute setup at smith.langchain.com |
| 1–2 | Install & Concepts | `LANGCHAIN_TRACING_V2=true` activates everything |
| 3 | Auto-tracing | Zero code change — every LangGraph call is traced |
| 4 | `@traceable` | Wrap pure-Python functions as custom spans |
| 5 | Cyclic Agent | Each loop iteration is a separate child run |
| 6 | Memory Graph | Use `run_name` + `thread_id` to group multi-turn sessions |
| 7 | HITL | Two-phase traces — pause + resume — full audit trail |
| 8 | Loan Approval | `tags` + `metadata` for production filtering |
| 9 | Feedback | Attach `correctness`, `relevance` scores to runs |
| 10 | Dashboard | Timeline, Tokens, Feedback, Child Runs panels |
| 11 | Datasets | Save examples for regression testing |
| 12 | Evaluation | Run custom evaluators across entire dataset |
| 13 | Latency & Cost | Token counting, cost estimation, SDK run queries |
| 14 | Error Tracking | Failed nodes captured with full input context |

---

### Production Readiness Checklist

```
✅  LANGCHAIN_TRACING_V2=true in .env (or environment)
✅  LANGSMITH_API_KEY set and validated
✅  LANGCHAIN_PROJECT set to a meaningful project name
✅  Every graph.invoke() call has run_name in config
✅  Business-important runs have tags and metadata
✅  @traceable added to critical non-LangChain functions
✅  Feedback attached to runs after human review
✅  Dataset created with golden examples
✅  Evaluation experiment run before each model upgrade
✅  Error filter in LangSmith bookmarked for daily review
```

---

### LangSmith Best Practices

| Practice | Why |
|----------|-----|
| Use `run_name` on every invoke | Makes runs searchable by name |
| Add `tags` for task categories | Filter `loan-approval`, `qa-pipeline` etc |
| Add `metadata` with user/session IDs | Essential for per-user debugging |
| Always check for `ls_client is None` | Graceful degradation when key missing |
| Use `@traceable` on pre/post-processing | Captures the full picture, not just LLM calls |
| Create a dataset early | Start saving examples from day 1 of production |
| Run evaluations on every prompt change | Catch regressions before users do |
| Review error runs weekly | Identify recurring failure patterns |

---

### Quick Reference: Key LangSmith Patterns

```python
# ─── 1. Enable tracing (in .env) ─────────────────────────────────────────
LANGCHAIN_TRACING_V2=true
LANGSMITH_API_KEY=lsv2_pt_...
LANGCHAIN_PROJECT=my-project

# ─── 2. Named trace with metadata ────────────────────────────────────────
graph.invoke(state, config={
    "run_name": "my-run-name",
    "tags": ["category-a"],
    "metadata": {"user_id": "u001"},
})

# ─── 3. @traceable custom span ───────────────────────────────────────────
from langsmith import traceable

@traceable(name="my_span", run_type="chain")
def my_function(input_data):
    return result

# ─── 4. Attach feedback to a run ─────────────────────────────────────────
ls_client.create_feedback(run_id=run_id, key="correctness", score=1.0)

# ─── 5. Create a dataset + run evaluation ────────────────────────────────
dataset = ls_client.create_dataset("my-dataset")
ls_client.create_examples(inputs=[...], outputs=[...], dataset_id=dataset.id)
from langsmith.evaluation import evaluate
results = evaluate(my_app, data="my-dataset", evaluators=[my_evaluator])
```

---

> **Next Steps:**
> 1. Add `run_name` and `metadata` to every `graph.invoke()` call in your production code
> 2. Create a dataset from your most common user queries
> 3. Set up a weekly evaluation run to track model quality over time
> 4. Review LangSmith docs: https://docs.smith.langchain.com